# 02. Hipoteza 1: Eskalacja emocjonalna

Sprawdzamy, czy link negatywny jest częstszy, gdy w poprzednich 24 godzinach ta sama para `SOURCE_SUBREDDIT -> TARGET_SUBREDDIT` miała wysokie natężenie `LIWC_Anger`.


In [1]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "database" / "NajnowszaWersjaBazy1205.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_PATH exists:", DATA_PATH.exists())


PROJECT_ROOT: C:\Users\szymon\projekt_reddit
DATA_PATH exists: True


In [2]:
from scipy import stats
from src.reddit_pbl.features import add_pair_history_features, sentiment_to_binary_negative


## Metoda

1. Sortujemy rekordy po czasie.
2. Dla każdej skierowanej pary subredditów liczymy wcześniejsze interakcje z ostatnich 24 godzin.
3. Wysoki anger definiujemy jako średni wcześniejszy `LIWC_Anger` co najmniej na poziomie 75 percentyla wśród rekordów z jakąkolwiek wcześniejszą interakcją.
4. Porównujemy odsetek negatywnych linków i liczymy test Fishera oraz iloraz szans.


In [3]:
df = pd.read_csv(DATA_PATH)
df["TIMESTAMP"] = pd.to_datetime(df["TIMESTAMP"])

features = add_pair_history_features(df)
features["is_negative_link"] = sentiment_to_binary_negative(features["LINK_SENTIMENT"])

history_rows = features[features["prev_pair_interactions_24h"] > 0].copy()
anger_threshold = history_rows["prev_pair_mean_anger_24h"].quantile(0.75)

features["high_previous_anger_24h"] = (
    (features["prev_pair_interactions_24h"] > 0)
    & (features["prev_pair_mean_anger_24h"] >= anger_threshold)
)

table = pd.crosstab(features["high_previous_anger_24h"], features["is_negative_link"])
table = table.reindex(index=[False, True], columns=[0, 1], fill_value=0)
display(table)

odds_ratio, fisher_p = stats.fisher_exact(table.to_numpy())
chi2, chi2_p, _, _ = stats.chi2_contingency(table.to_numpy())
print("Próg wysokiego anger:", anger_threshold)
print("Odds ratio:", odds_ratio)
print("Fisher p-value:", fisher_p)
print("Chi2 p-value:", chi2_p)


is_negative_link,0,1
high_previous_anger_24h,,
False,45950,3844
True,114,10


Próg wysokiego anger: 0.0064516129032258
Odds ratio: 1.0485696551472334
Fisher p-value: 0.8656127056346432
Chi2 p-value: 1.0


In [4]:
rates = (
    features.groupby("high_previous_anger_24h")["is_negative_link"]
    .agg(rows="size", negative_links="sum", negative_rate="mean")
    .reset_index()
)
display(rates)

rates.to_csv(OUTPUT_DIR / "notebook_h1_rates.csv", index=False)


,high_previous_anger_24h,rows,negative_links,negative_rate
0,False,49794,3844,0.077198
1,True,124,10,0.080645


## Wniosek

W uruchomionej analizie rdzeniowej wynik nie wspiera H1: grupa z wysokim wcześniejszym `LIWC_Anger` ma bardzo podobny odsetek linków negatywnych do pozostałych rekordów, a test Fishera nie wskazuje istotnej różnicy.

Interpretacyjnie ważne jest też to, że tylko niewielka liczba rekordów ma historię tej samej pary w poprzednich 24 godzinach. Hipotezę warto powtórzyć także dla okien 48h i 7 dni albo dla par nieskierowanych.
